In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_sales_data
from src.preprocessing.sales import prepare_sales_data
from src.features.time_features import create_time_features
from src.features.lag_features import create_lag_features
from src.features.rolling_features import create_rolling_features
from src.preprocessing.splitting import time_based_split
from src.evaluation.metrics import calculate_mae
from src.models.forecasting import train_model
from src.evaluation.cross_validation import time_series_cv_mae

In [2]:
df = load_sales_data("../data/raw/sales.csv")

df.head()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,units_sold
0,2024-01-01,P001,Wireless Headphones,Electronics,1999.0,0,0,0,1,False,33
1,2024-01-02,P001,Wireless Headphones,Electronics,1999.0,0,0,1,1,False,39
2,2024-01-03,P001,Wireless Headphones,Electronics,1999.0,0,0,2,1,False,39
3,2024-01-04,P001,Wireless Headphones,Electronics,1799.1,10,1,3,1,False,59
4,2024-01-05,P001,Wireless Headphones,Electronics,1999.0,0,0,4,1,False,35


In [3]:
df = prepare_sales_data(df)

df.head()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,units_sold
0,2024-01-01,P001,Wireless Headphones,Electronics,1999.0,0,0,0,1,False,33
1,2024-01-02,P001,Wireless Headphones,Electronics,1999.0,0,0,1,1,False,39
2,2024-01-03,P001,Wireless Headphones,Electronics,1999.0,0,0,2,1,False,39
3,2024-01-04,P001,Wireless Headphones,Electronics,1799.1,10,1,3,1,False,59
4,2024-01-05,P001,Wireless Headphones,Electronics,1999.0,0,0,4,1,False,35


In [4]:
df = create_time_features(df)

In [5]:
df = create_lag_features(df)

In [6]:
df = create_rolling_features(df)

In [7]:
print(df["lag_14"])

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
        ... 
3650    49.0
3651    84.0
3652    37.0
3653    33.0
3654    44.0
Name: lag_14, Length: 3655, dtype: float64


# ==========================================
# TRAIN / VALIDATION / TEST SPLIT
# ==========================================

In [8]:
train, validation, test = time_based_split(
    df,
    train_end="2025-07-01",
    validation_end="2025-10-01",
)

In [9]:
print(
    "Train:",
    train["date"].min(),
    "to",
    train["date"].max()
)

print(
    "Validation:",
    validation["date"].min(),
    "to",
    validation["date"].max()
)

print(
    "Test:",
    test["date"].min(),
    "to",
    test["date"].max()
)

Train: 2024-01-01 00:00:00 to 2025-06-30 00:00:00
Validation: 2025-07-01 00:00:00 to 2025-09-30 00:00:00
Test: 2025-10-01 00:00:00 to 2025-12-31 00:00:00


In [10]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (2735, 22)
Validation: (460, 22)
Test: (460, 22)


Baseline model testing 

In [11]:
from src.models.baselines import naive_forecast

baseline = naive_forecast(df)

In [12]:
baseline_test = baseline[
    baseline["date"] >= "2025-10-01"
].dropna()

In [13]:
baseline_mae = calculate_mae(
    baseline_test["units_sold"],
    baseline_test["prediction"]
)

print(
    f"Naive Baseline MAE: {baseline_mae:.2f}"
)

Naive Baseline MAE: 10.10


Random Forest

In [14]:
baseline.head(10)

,date,product_id,units_sold,prediction
0,2024-01-01,P001,33,NaN
1,2024-01-02,P001,39,33.0
2,2024-01-03,P001,39,39.0
3,2024-01-04,P001,59,39.0
4,2024-01-05,P001,35,59.0
5,2024-01-06,P001,41,35.0
6,2024-01-07,P001,30,41.0
7,2024-01-08,P001,42,30.0
8,2024-01-09,P001,43,42.0
9,2024-01-10,P001,34,43.0


In [15]:
features = [
    "price",
    "discount",
    "promotion",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
]

In [16]:
X_train = train[features]
y_train = train["units_sold"]

X_validation = validation[features]
y_validation = validation["units_sold"]

X_test = test[features]
y_test = test["units_sold"]

In [17]:
X_train.isna().sum()

price                0
discount             0
promotion            0
day_of_week          0
month                0
is_weekend           0
lag_1                5
lag_7               35
lag_14              70
lag_28             140
rolling_mean_7      35
rolling_mean_14     70
rolling_mean_28    140
rolling_std_7       35
dtype: int64

In [18]:
train_ml = train.dropna(subset=features).copy()
validation_ml = validation.dropna(subset=features).copy()
test_ml = test.dropna(subset=features).copy()

In [19]:
X_train = train_ml[features]
y_train = train_ml["units_sold"]

X_validation = validation_ml[features]
y_validation = validation_ml["units_sold"]

X_test = test_ml[features]
y_test = test_ml["units_sold"]

In [20]:
from sklearn.ensemble import RandomForestRegressor

In [21]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


In [22]:
model = train_model(
    model,
    X_train,
    y_train
)

In [23]:
validation_predictions = model.predict(
    X_validation
)

mae = calculate_mae(
    y_validation,
    validation_predictions
)

print(
    f"Validation MAE: {mae:.2f}"
)

Validation MAE: 3.94


Seasonal Naive

In [24]:
print(train_ml[["date", "product_id", "units_sold"]].head(10))

print(validation_ml[["date", "product_id", "units_sold"]].head(10))

         date product_id  units_sold
28 2024-01-29       P001          40
29 2024-01-30       P001          58
30 2024-01-31       P001          51
31 2024-02-01       P001          37
32 2024-02-02       P001          52
33 2024-02-03       P001          51
34 2024-02-04       P001          52
35 2024-02-05       P001          42
36 2024-02-06       P001          55
37 2024-02-07       P001          55
          date product_id  units_sold
547 2025-07-01       P001          30
548 2025-07-02       P001          42
549 2025-07-03       P001          45
550 2025-07-04       P001          35
551 2025-07-05       P001          37
552 2025-07-06       P001          36
553 2025-07-07       P001          30
554 2025-07-08       P001          43
555 2025-07-09       P001          35
556 2025-07-10       P001          33


In [25]:
print(train_ml["date"].min(), train_ml["date"].max())
print(validation_ml["date"].min(), validation_ml["date"].max())


2024-01-29 00:00:00 2025-06-30 00:00:00
2025-07-01 00:00:00 2025-09-30 00:00:00


In [26]:
historical_data = pd.concat(
    [train_ml, validation_ml]
).sort_values(
    ["product_id", "date"]
)

In [27]:
historical_data["seasonal_naive_prediction"] = (
    historical_data
    .groupby("product_id")["units_sold"]
    .shift(7)
)

In [28]:
seasonal_validation = historical_data[
    historical_data["date"].isin(validation_ml["date"])
]

In [29]:
seasonal_predictions = seasonal_validation[
    "seasonal_naive_prediction"
]

In [30]:
seasonal_actual = seasonal_validation[
    "units_sold"
]

In [31]:
seasonal_mae = calculate_mae(
    seasonal_actual,
    seasonal_predictions
)

print(f"Seasonal Naive MAE: {seasonal_mae:.2f}")

Seasonal Naive MAE: 9.49


In [32]:
print(
    historical_data[
        ["date", "product_id", "units_sold", "seasonal_naive_prediction"]
    ].tail(15)
)

           date product_id  units_sold  seasonal_naive_prediction
3548 2025-09-16       P005          34                       37.0
3549 2025-09-17       P005          33                       46.0
3550 2025-09-18       P005          41                       51.0
3551 2025-09-19       P005          33                       32.0
3552 2025-09-20       P005          31                       48.0
3553 2025-09-21       P005          48                       61.0
3554 2025-09-22       P005          53                       37.0
3555 2025-09-23       P005          36                       34.0
3556 2025-09-24       P005          32                       33.0
3557 2025-09-25       P005          39                       41.0
3558 2025-09-26       P005          32                       33.0
3559 2025-09-27       P005          37                       31.0
3560 2025-09-28       P005          57                       48.0
3561 2025-09-29       P005          48                       53.0
3562 2025-

Moving Average 

In [33]:
historical_data["moving_average_prediction"] = (
    historical_data
    .groupby("product_id")["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

In [34]:
print(
    historical_data[
        ["date", "product_id", "units_sold", "moving_average_prediction"]
    ].tail(15)
)

           date product_id  units_sold  moving_average_prediction
3548 2025-09-16       P005          34                  44.571429
3549 2025-09-17       P005          33                  44.142857
3550 2025-09-18       P005          41                  42.285714
3551 2025-09-19       P005          33                  40.857143
3552 2025-09-20       P005          31                  41.000000
3553 2025-09-21       P005          48                  38.571429
3554 2025-09-22       P005          53                  36.714286
3555 2025-09-23       P005          36                  39.000000
3556 2025-09-24       P005          32                  39.285714
3557 2025-09-25       P005          39                  39.142857
3558 2025-09-26       P005          32                  38.857143
3559 2025-09-27       P005          37                  38.714286
3560 2025-09-28       P005          57                  39.571429
3561 2025-09-29       P005          48                  40.857143
3562 2025-

In [35]:
moving_average_validation = historical_data[
    historical_data["date"].isin(validation_ml["date"])
]

In [36]:
moving_average_validation.info()

<class 'pandas.DataFrame'>
Index: 460 entries, 547 to 3562
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       460 non-null    datetime64[us]
 1   product_id                 460 non-null    str           
 2   product_name               460 non-null    str           
 3   category                   460 non-null    str           
 4   price                      460 non-null    float64       
 5   discount                   460 non-null    int64         
 6   promotion                  460 non-null    int64         
 7   day_of_week                460 non-null    int32         
 8   month                      460 non-null    int32         
 9   is_weekend                 460 non-null    bool          
 10  units_sold                 460 non-null    int64         
 11  year                       460 non-null    int32         
 12  day                  

In [37]:
moving_average_mae = calculate_mae(
    moving_average_validation["units_sold"],
    moving_average_validation["moving_average_prediction"]
)

print(f"Moving Average MAE: {moving_average_mae:.2f}")

Moving Average MAE: 7.35


Linear Regression 

In [38]:
from sklearn.linear_model import LinearRegression


linear_model = LinearRegression()


linear_model = train_model(linear_model, X_train, y_train)

linear_predictions = linear_model.predict(
    X_validation
)


linear_mae = calculate_mae(
    y_validation,
    linear_predictions
)

print(f"Linear Regression MAE: {linear_mae:.2f}")

Linear Regression MAE: 4.20


Gradient Boosting  

In [39]:
from sklearn.ensemble import GradientBoostingRegressor

In [40]:
gradient_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [41]:

gradient_model = train_model(gradient_model, X_train, y_train)

In [42]:
gradient_predictions = gradient_model.predict(
    X_validation
)

In [43]:
gradient_mae = calculate_mae(
    y_validation,
    gradient_predictions
)

print(f"Gradient Boosting MAE: {gradient_mae:.2f}")

Gradient Boosting MAE: 3.67


XGBoost

In [44]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    n_jobs=-1
)

In [45]:
xgb_model = train_model(xgb_model, X_train, y_train)

In [46]:
xgb_predictions = xgb_model.predict(
    X_validation
)

In [47]:
xgb_mae = calculate_mae(
    y_validation,
    xgb_predictions
)

print(f"XGBoost MAE: {xgb_mae:.2f}")

XGBoost MAE: 3.65


Time series cross validation 

In [48]:
tscv = TimeSeriesSplit(n_splits=4)

for fold, (train_index, validation_index) in enumerate(
    tscv.split(X_train),
    start=1
):
    print(f"Fold {fold}")
    print("Training:", train_index[0], "→", train_index[-1])
    print("Validation:", validation_index[0], "→", validation_index[-1])
    print()

Fold 1
Training: 0 → 518
Validation: 519 → 1037

Fold 2
Training: 0 → 1037
Validation: 1038 → 1556

Fold 3
Training: 0 → 1556
Validation: 1557 → 2075

Fold 4
Training: 0 → 2075
Validation: 2076 → 2594



In [49]:
train_ml = train_ml.sort_values(
    ["date", "product_id"]
).reset_index(drop=True)

In [50]:
X_train = train_ml[features]
y_train = train_ml["units_sold"]

In [51]:
tscv = TimeSeriesSplit(n_splits=4)

In [52]:
for fold, (train_index, validation_index) in enumerate(
    tscv.split(X_train),
    start=1
):
    print(f"Fold {fold}")

    print(
        "Training:",
        train_ml.iloc[train_index]["date"].iloc[0],
        "→",
        train_ml.iloc[train_index]["date"].iloc[-1]
    )

    print(
        "Validation:",
        train_ml.iloc[validation_index]["date"].iloc[0],
        "→",
        train_ml.iloc[validation_index]["date"].iloc[-1]
    )

    print()

Fold 1
Training: 2024-01-29 00:00:00 → 2024-05-11 00:00:00
Validation: 2024-05-11 00:00:00 → 2024-08-23 00:00:00

Fold 2
Training: 2024-01-29 00:00:00 → 2024-08-23 00:00:00
Validation: 2024-08-23 00:00:00 → 2024-12-05 00:00:00

Fold 3
Training: 2024-01-29 00:00:00 → 2024-12-05 00:00:00
Validation: 2024-12-05 00:00:00 → 2025-03-19 00:00:00

Fold 4
Training: 2024-01-29 00:00:00 → 2025-03-19 00:00:00
Validation: 2025-03-19 00:00:00 → 2025-06-30 00:00:00



In [53]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    n_jobs=-1
)

fold_maes = time_series_cv_mae(
    model,
    X_train,
    y_train,
    n_splits=4
)

for fold, mae in enumerate(fold_maes, start=1):
    print(f"Fold {fold} MAE: {mae:.2f}")

print(
    f"\nAverage CV MAE: {sum(fold_maes) / len(fold_maes):.2f}"
)

Fold 1 MAE: 5.20
Fold 2 MAE: 3.68
Fold 3 MAE: 4.72
Fold 4 MAE: 4.46

Average CV MAE: 4.51


In [54]:
from xgboost import XGBRegressor

tscv = TimeSeriesSplit(n_splits=4)

results = []

for n_estimators in [100, 200, 300]:

    for learning_rate in [0.03, 0.05, 0.1]:

        for max_depth in [2, 3, 4]:

            fold_maes = []

            for train_index, validation_index in tscv.split(X_train):

                X_fold_train = X_train.iloc[train_index]
                y_fold_train = y_train.iloc[train_index]

                X_fold_validation = X_train.iloc[validation_index]
                y_fold_validation = y_train.iloc[validation_index]

                model = XGBRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    random_state=42,
                    n_jobs=-1
                )

                model = train_model(model, X_fold_train, y_fold_train)

                predictions = model.predict(
                    X_fold_validation
                )

                mae = calculate_mae(
                    y_fold_validation,
                    predictions
                )

                fold_maes.append(mae)

            average_mae = sum(fold_maes) / len(fold_maes)

            results.append({
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "cv_mae": average_mae
            })

print("Tuning complete!")

Tuning complete!


In [55]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "cv_mae"
).reset_index(drop=True)

print(results_df)

    n_estimators  learning_rate  max_depth    cv_mae
0            200           0.05          2  4.488059
1            300           0.03          2  4.495578
2            300           0.05          2  4.495653
3            100           0.10          2  4.506107
4            300           0.03          3  4.512997
5            200           0.05          3  4.513877
6            200           0.03          3  4.519336
7            100           0.10          3  4.526137
8            100           0.05          3  4.543914
9            200           0.10          2  4.555561
10           300           0.05          3  4.555605
11           200           0.03          2  4.569616
12           200           0.03          4  4.579629
13           200           0.05          4  4.593056
14           300           0.03          4  4.597198
15           100           0.05          4  4.603944
16           200           0.10          3  4.623246
17           300           0.05          4  4.

In [56]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=2,
        random_state=42,
        n_jobs=-1
    )
}

cv_results = []

for model_name, model in models.items():

    fold_maes = []

    for train_index, validation_index in tscv.split(X_train):

        X_fold_train = X_train.iloc[train_index]
        y_fold_train = y_train.iloc[train_index]

        X_fold_validation = X_train.iloc[validation_index]
        y_fold_validation = y_train.iloc[validation_index]

        model = train_model(model, X_fold_train, y_fold_train)

        predictions = model.predict(
            X_fold_validation
        )

        mae = calculate_mae(
            y_fold_validation,
            predictions
        )

        fold_maes.append(mae)

    average_mae = sum(fold_maes) / len(fold_maes)

    cv_results.append({
        "model": model_name,
        "cv_mae": average_mae
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    "cv_mae"
).reset_index(drop=True)

print(cv_results_df)

               model    cv_mae
0            XGBoost  4.488059
1  Gradient Boosting  4.571241
2      Random Forest  4.692148


In [57]:
final_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    random_state=42,
    n_jobs=-1
)

final_xgb = train_model(final_xgb, X_train, y_train)

In [58]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": final_xgb.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance)

            feature  importance
0          discount    0.257716
1   rolling_mean_28    0.254769
2   rolling_mean_14    0.175561
3    rolling_mean_7    0.130206
4             price    0.055039
5       day_of_week    0.045159
6             lag_7    0.037050
7            lag_14    0.016441
8             month    0.012745
9             lag_1    0.005906
10           lag_28    0.005472
11    rolling_std_7    0.003936
12       is_weekend    0.000000
13        promotion    0.000000


Final testing 

In [59]:
X_final_train = pd.concat(
    [X_train, X_validation]
)

y_final_train = pd.concat(
    [y_train, y_validation]
)

In [60]:
final_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    random_state=42,
    n_jobs=-1
)

In [61]:
final_model = train_model(final_model, X_final_train, y_final_train)

In [62]:
test_predictions = final_model.predict(
    X_test
)

In [63]:
test_mae = calculate_mae(
    y_test,
    test_predictions
)

print(f"Final Test MAE: {test_mae:.2f}")

Final Test MAE: 3.68


Error Analysis 

In [64]:
test_results = test_ml[[
    "date",
    "product_id",
    "units_sold"
]].copy()

test_results["prediction"] = test_predictions

test_results["error"] = (
    test_results["units_sold"]
    - test_results["prediction"]
)

test_results["absolute_error"] = (
    test_results["error"].abs()
)

print(test_results.head(10))

          date product_id  units_sold  prediction      error  absolute_error
639 2025-10-01       P001          39   45.356434  -6.356434        6.356434
640 2025-10-02       P001          27   38.465065 -11.465065       11.465065
641 2025-10-03       P001          45   51.086365  -6.086365        6.086365
642 2025-10-04       P001          40   43.227623  -3.227623        3.227623
643 2025-10-05       P001          35   36.125446  -1.125446        1.125446
644 2025-10-06       P001          36   31.486355   4.513645        4.513645
645 2025-10-07       P001          29   38.995300  -9.995300        9.995300
646 2025-10-08       P001          40   37.202240   2.797760        2.797760
647 2025-10-09       P001          46   41.942005   4.057995        4.057995
648 2025-10-10       P001          53   53.471867  -0.471867        0.471867


In [65]:
worst_predictions = test_results.sort_values(
    "absolute_error",
    ascending=False
)

print(worst_predictions.head(10))

           date product_id  units_sold  prediction      error  absolute_error
3637 2025-12-14       P005          84   61.811047  22.188953       22.188953
3629 2025-12-06       P005          89   67.162621  21.837379       21.837379
3652 2025-12-29       P005          24   40.935738 -16.935738       16.935738
2185 2025-12-24       P003          33   47.843716 -14.843716       14.843716
662  2025-10-24       P001          64   49.361115  14.638885       14.638885
672  2025-11-03       P001          63   49.588108  13.411892       13.411892
3641 2025-12-18       P005          51   64.215942 -13.215942       13.215942
685  2025-11-16       P001          76   62.961456  13.038544       13.038544
1437 2025-12-07       P002          60   47.071613  12.928387       12.928387
730  2025-12-31       P001          24   36.492268 -12.492268       12.492268


In [66]:
product_error = (
    test_results
    .groupby("product_id")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
)

print(product_error)

product_id
P005    4.758361
P001    4.547421
P003    3.861725
P002    2.978732
P004    2.268127
Name: absolute_error, dtype: float64


In [67]:
average_error = (
    test_results
    .groupby("product_id")["error"]
    .mean()
    .sort_values()
)

print(average_error)

product_id
P002   -0.472225
P004   -0.334454
P001    0.222788
P005    0.353435
P003    0.391681
Name: error, dtype: float64
